(closedorbit)=
# (GPU-)Batched Closed Orbit Finder

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

The `find_closed_orbit` function can be used to find the fixed point of a periodic beamline. It uses 
the Newton  method implemented in the [`BatchSolve`](https://github.com/mattsignorelli/BatchSolve.jl) 
package. By default, automatic differentiation via [`ForwardDiff`](https://github.com/JuliaDiff/ForwardDiff.jl) 
is used to compute the Jacobian(s) each iteration.

In [2]:
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    d = sb.Drift(L=1.2)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)
    kick = sb.HKicker(Kn0L=1e-5)

fodo = sb.Beamline([qf, d, qd, d, kick],
                   species_ref=sb.Species("electron"), E_ref=18e9)  # some Beamline
co_sol = sb.find_closed_orbit(fodo)

A `NamedTuple` is returned containing the properties:

- `v0`: The closed orbit
- `coasting_beam`: `True` if no longitudinal oscillations (4D closed orbit), `False` otherwise
- `sol`: Another `NamedTuple` with details of the solution from `BatchSolve`:
  - `u`: The coordinates the Newton method solved for: all six, or for a coasting beam only the
    four transverse ones (δ is fixed), one row per particle.
  - `f`: The final residual: the change of these coordinates over one turn, which is zero on the
    closed orbit.
  - `jac`: The Jacobian of the residual at the last iteration (of all particles together).
  - `retcode`: `0x0` if the solution converged, `0x1` if the solver failed, `0x2` if it reached the
    maximum number of iterations (one per particle with several particles).
  - `iters`: The number of iterations taken (one per particle with several particles).

An initial "guess" for the closed orbit can be provided by specifying a `1 x 6` matrix of
particle phase space coordinates to the keyword argument `v0`. By default, it is
`zeros(1, 6)`.

In [3]:
co_sol = sb.find_closed_orbit(fodo, v0=[[1e-5, 2e-6, -1e-5, -2e-6, 0.0, 1e-5]])
co_sol.v0

array([[-4.98980961e-04, -4.96045976e-05, -1.63063314e-18,
         1.45666480e-19,  0.00000000e+00,  1.00000000e-05]])

This beamline has no RF, so the beam is coasting (`co_sol.coasting_beam` is `True`), and the
last coordinate δ of the guess did not change: the result is the closed orbit for δ = 1e-5
(see δ-Dependent Closed Orbits below).

:::{note}
When `v0` is provided, it **will be mutated in place with the result!**
:::

## δ-Dependent Closed Orbits

For beamlines with coasting beam, δ-dependent closed orbits can be computed. To do this, we
set `v0` equal to a matrix of `n_particles x 6`:

In [4]:
# To get delta-dependent closed orbits:
v0 = [[0, 0, 0, 0, 0, 0.1e-2],  # δ = 0.1e-2
      [0, 0, 0, 0, 0, 0.2e-2]]  # δ = 0.2e-2

co_sol = sb.find_closed_orbit(fodo, v0=v0)
co_sol.v0

array([[-4.99429581e-04, -4.96448475e-05, -1.63386271e-18,
         1.45956046e-19,  0.00000000e+00,  1.00000000e-03],
       [-4.99882729e-04, -4.96855039e-05, -1.63712909e-18,
         1.46248919e-19,  0.00000000e+00,  2.00000000e-03]])

If `v0` is a CUDA array type `CuArray`{pytext}` (sb.device("cuda").asarray(...) in Python)`, then
the Newton method used to find the closed
orbits will be GPU-parallelized, with bindings directly to cuBLAS's batched linear system
solvers.


```python
gpu = sb.device("cuda")
n_particles = 100000
v0 = np.zeros((n_particles, 6))
v0[:, 5] = np.linspace(0, 1e-2, n_particles)

co_sol = sb.find_closed_orbit(fodo, v0=gpu.asarray(v0))
```

## Closed Orbits with Batch Parameters

If any element parameters are defined with [`BatchParam`s](batch.md), then as with `track`,
each particle in the initial guess matrix will see that corresponding parameter (with a guess of
one row, the default `v0`, the first value). Element lengths and the reference energy can be
batched too.

Below, we first find two closed orbits, one for `kick.Kn0L=1e-5` and one for `kick.Kn0L=2e-5`,
and then the same with `δ = 0.1e-2` for the first and `δ = 0.2e-2` for the second:

In [5]:
# With BatchParams
kick.Kn0L = sb.BatchParam([1e-5, 2e-5])
v0 = np.zeros((2, 6))
co_sol = sb.find_closed_orbit(fodo, v0=v0)

# BatchParams + delta-dependent closed orbits:
v0 = [[0, 0, 0, 0, 0, 0.1e-2],  # δ = 0.1e-2
      [0, 0, 0, 0, 0, 0.2e-2]]  # δ = 0.2e-2
co_sol = sb.find_closed_orbit(fodo, v0=v0)
co_sol

(v0 = [-0.000499429580692261 -4.964484747922549e-5 … 0.0 0.001; -0.0009997640972285228 -9.937089270145935e-5 … 0.0 0.002], coasting_beam = true, sol = (u = [-0.000499429580692261 -4.964484747922549e-5 -1.6338627084277428e-18 1.4595604602569772e-19; -0.0009997640972285228 -9.937089270145935e-5 -3.2742409184428352e-18 2.924962167559868e-19], f = [-1.0842021724855044e-19 0.0 -7.703719777548943e-34 1.6851887013388314e-34; 0.0 8.131516293641283e-20 -2.311115933264683e-33 9.62964972193618e-35], jac = sparse([1, 3, 5, 7, 2, 4, 6, 8, 1, 3  …  6, 8, 1, 3, 5, 7, 2, 4, 6, 8], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  6, 6, 7, 7, 7, 7, 8, 8, 8, 8], [0.3729345122731341, 0.04962649592106685, -8.118329739482383e-17, -5.584159974607817e-17, 0.37249557502755526, 0.04957706733496511, -8.113510977327135e-17, -5.586486137515083e-17, -3.7517388268279843, -0.2978139673475231  …  -0.2381508356712967, 0.04957706045998415, -2.3231047991603536e-16, -1.6157484593922657e-16, -3.0189641981835194, 0.31344220382690424, -2.3

Again, by making these `CuArray`s, the closed orbit finding will be GPU-parallelized.


```python
gpu = sb.device("cuda")

n_kick_strengths = 100
n_deltas = 1000
n_particles = n_kick_strengths * n_deltas
v0 = np.zeros((n_particles, 6))
v0[:, 5] = np.tile(np.linspace(0, 1e-2, n_deltas), n_kick_strengths)

kick_strengths = np.repeat(np.linspace(1e-5, 2e-5, n_kick_strengths), n_deltas)
kick.Kn0L = sb.BatchParam(gpu.asarray(kick_strengths))
co_sol = sb.find_closed_orbit(fodo, v0=gpu.asarray(v0))
```

```{docstring} find_closed_orbit
```